# Step 0：A公司煤炭数据治理项目 — 背景与数据说明

## 1. 项目背景

A公司是一家大型煤炭能源集团，下辖 **5座矿井**，年产原煤约 **3000万吨**。集团总部设有安全生产、调度指挥、煤质管控、财务结算等多个业务中心，各中心依赖信息系统进行日常决策。

## 2. 数据治理驱动力

| 驱动力 | 具体表现 |
|--------|----------|
| **安全生产合规** | 瓦斯、一氧化碳等监测数据需保存≥2年，异常告警须可追溯 |
| **经营分析需求** | 产销一体化分析需要打通ERP订单与PI生产实绩 |
| **数据孤岛严重** | SAP-ERP、PI-System、SCADA、LIMS、OA五个系统相互独立，无统一数据标准 |
| **质量问题频发** | 各系统数据质量问题频发（空值、重复、格式不一致），影响分析准确性 |
| **监管报送压力** | 煤质数据、产量数据需定期向能源局、应急管理部报送 |


本项目模拟了 A公司真实部署的 5 个异构信息系统，它们各自独立运行、数据标准不统一，构成了典型的「数据孤岛」场景。

- **SAP-ERP**（Oracle关系型）：客户主数据 + 销售订单（VBAK/VBAP/KNA1）
- **PI-System**（TimescaleDB时序）：100标签时序传感器数据
- **LIMS**（SQL Server）：煤质化验批次数据
- **OA**（流程引擎）：审批流程记录
- **SCADA**（Kafka）：设备实时状态监控


---

## 3. 各系统数据问题识别和治理策略

在探查数据之前，先了解每张表的字段结构，便于理解后续质量问题的影响范围。每个系统均按 **表结构说明 → 数据预览 → 问题识别 → 治理策略** 的顺序展开。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Noto Sans CJK JP 包含中文字形，Noto Sans 回退西文
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK JP', 'Noto Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

import os
import glob

# 使用绝对路径，确保无论从哪个目录启动 kernel 都能正确定位
DATA_ROOT = '/home/szs/Playground/dg-demo/data/historical'
print(f'数据路径: {DATA_ROOT}')


数据路径: /home/szs/Playground/dg-demo/data/historical


### 3.1 SAP-ERP 表结构说明

**KNA1 — 客户主数据**

| 字段名 | 含义 | 关键业务含义 |
|--------|------|-------------|
| KUNNR | 客户编号（主键，6位） | ERP中唯一客户标识，用于所有销售/财务交易关联 |
| NAME1 | 客户名称（主名称） | 下游报表展示主名称，须规范命名 |
| NAME2 | 客户名称（副名称/分支机构） | 分支机构名称，去重时应合并 |
| ORT01 | 城市 | 地址分析维度 |
| STCD1 | 统一社会信用代码（营业执照号，18位） | 工商核验字段，须为18位统一社会信用代码，格式错误影响合规审查 |
| ERDAT | 创建日期 | 客户建档时间，用于客户生命周期分析 |

**VBAK — 销售订单抬头表**

| 字段名 | 含义 | 关键业务含义 |
|--------|------|-------------|
| VBELN | 销售订单号 | 销售订单唯一编号 |
| ERDAT | 订单创建日期 | 订单日期，用于产销分析时间维度 |
| ERZET | 订单创建时间 | 订单时间 |
| ERNAM | 创建人 | 创建操作员 |
| AUART | 订单类型 | OR=标准订单，ZOR=出口订单等 |
| KUNNR | 客户编号 | 关联客户主数据，关联失效则订单无效 |
| NETWR | 净价合计（金额） | 含税金额，财务结算核心字段，空值影响收入统计 |
| WAERK | 币种 | CNY/RMB |
| BZIRK | 销售区域 | 销售区域编码 |
| VKORG | 销售组织 | 区分业务单元 |
| VTWEG | 分销渠道 | 分销渠道 |
| SPART | 产品组 | 产品组 |
| BSTNK | 客户采购订单号 | 客户侧采购单号，用于对账 |
| FABKL | 工厂日历 | 关联矿井 |
| LIFSK | 交货状态 | 影响发货分析 |
| FAKSK | 开票状态 | 影响收入确认 |

**VBAP — 销售订单行项目表**

| 字段名 | 含义 | 关键业务含义 |
|--------|------|-------------|
| VBELN | 销售订单号 | 关联 VBAK 抬头，关联失效则行项目孤立 |
| POSNR | 行项目号 | 行项目序号，同一订单有多行 |
| MATNR | 物料编码 | 物料编码，关联产品主数据 |
| KWMENG | 数量 | 销售数量 |
| VRKME | 销售单位 | TO=吨，EA=件 |
| NETWR | 行项目净价 | 行项目金额 |
| WAERK | 币种 | 币种 |
| CHARG | 批次号 | 批次，溯源用 |
| WERKS | 工厂（矿井） | 工厂编码（CN01/CN02/CN03），对应具体矿井，跨系统关联关键字段 |
| LGORT | 库存地点 | 库存地点 |

#### 3.1.1 SAP-ERP 数据预览

In [2]:
# ── SAP-ERP ──
print("=" * 60)
print("【SAP-ERP】客户主数据 KNA1")
print("=" * 60)
kna1 = pd.read_parquet(os.path.join(DATA_ROOT, 'sap_erp', 'kna1.parquet'))
print(f"记录数: {len(kna1):,} | 列: {list(kna1.columns)}")
print(kna1.head(3).to_string(index=False))

print()
print("=" * 60)
print("【SAP-ERP】销售订单 VBAK (2023年样本)")
print("=" * 60)
vbak = pd.read_parquet(os.path.join(DATA_ROOT, 'sap_erp', 'vbak_year=2023.parquet'))
print(f"记录数: {len(vbak):,} | 列: {list(vbak.columns)}")
print(vbak.head(3).to_string(index=False))

print()
print("=" * 60)
print("【SAP-ERP】销售订单行项目 VBAP (2023年样本)")
print("=" * 60)
vbap = pd.read_parquet(os.path.join(DATA_ROOT, 'sap_erp', 'vbap_year=2023.parquet'))
print(f"记录数: {len(vbap):,} | 列: {list(vbap.columns)}")
print(vbap.head(3).to_string(index=False))

【SAP-ERP】客户主数据 KNA1
记录数: 15,000 | 列: ['KUNNR', 'NAME1', 'NAME2', 'ORT01', 'STCD1', 'ERDAT']
 KUNNR    NAME1 NAME2 ORT01          STCD1               ERDAT
100001 陕西煤业化工集团  分公司0    北京 91294966639924 2021-02-01T01:31:03
100002   大同煤矿集团  分公司1  呼和浩特 91373688976723 2022-10-28T15:24:58
100003 山西焦化能源集团  分公司2    西安 91329575994241 2021-02-02T10:09:01

【SAP-ERP】销售订单 VBAK (2023年样本)
记录数: 3,015,716 | 列: ['VBELN', 'ERDAT', 'ERZET', 'ERNAM', 'AUART', 'KUNNR', 'NETWR', 'WAERK', 'BZIRK', 'VKORG', 'VTWEG', 'SPART', 'BSTNK', 'FABKL', 'LIFSK', 'FAKSK']
     VBELN               ERDAT  ERZET     ERNAM AUART  KUNNR     NETWR WAERK BZIRK VKORG VTWEG SPART    BSTNK FABKL LIFSK FAKSK
1000000001 2023-02-07T06:48:14 061816   SAPUSER    OR 111374  53473.31   CNY  D004  CN01    10    00 PO452349  CN01            
1000000004 2023-09-20T16:15:45 083155     LIISI    OR 104826  36908.71   CNY  D002  CN02    20    01 PO526967  CN03            
1000000005 2023-07-23T19:36:36 063851 BATCH_JOB   ZOR 111130 371098.18   CNY 

#### 3.1.2 SAP-ERP 质量问题识别

In [3]:
# ── SAP-ERP 质量画像 ──────────────────────────────────────────────
print("=" * 60)
print("【SAP-ERP】全表质量画像 — KNA1/VBAK/VBAP")
print("=" * 60)

# 读取三张表
kna1 = pd.read_parquet(os.path.join(DATA_ROOT, 'sap_erp', 'kna1.parquet'))
vbak = pd.read_parquet(os.path.join(DATA_ROOT, 'sap_erp', 'vbak_year=2023.parquet'))
vbap = pd.read_parquet(os.path.join(DATA_ROOT, 'sap_erp', 'vbap_year=2023.parquet'))

print(f"KNA1: {len(kna1):,} 行 | VBAK: {len(vbak):,} 行 | VBAP: {len(vbap):,} 行")
print()

# ── KNA1 问题检测 ──────────────────────────────────────────────
kna1['STCD1'] = kna1['STCD1'].astype(str)
kna1['ERDAT_DT'] = pd.to_datetime(kna1['ERDAT'], errors='coerce')

dup_kunnr = kna1['KUNNR'].duplicated().sum()
stcd1_invalid = (~kna1['STCD1'].str.match(r'^\d{18}$')).sum()
name_short = (kna1['NAME1'].str.len() < 4).sum()
name_digit = (kna1['NAME1'].str.contains(r'\d', na=False)).sum()
erdat_bad = ((kna1['ERDAT_DT'] < '2015-01-01') | (kna1['ERDAT_DT'] > '2025-01-01')).sum()

print("【KNA1 客户主数据】问题检测结果")
print(f"  主键重复（KUNNR）: {dup_kunnr:,} 条 ({dup_kunnr/len(kna1)*100:.2f}%)")
print(f"  营业执照格式错误（STCD1 非18位数字）: {stcd1_invalid:,} 条 ({stcd1_invalid/len(kna1)*100:.2f}%)")
print(f"  名称过短（<4字符）: {name_short:,} 条")
print(f"  名称含数字: {name_digit:,} 条")
print(f"  创建日期越界: {erdat_bad:,} 条")
print()

# ── VBAK 问题检测 ──────────────────────────────────────────────
dup_vbak = vbak.duplicated().sum()
netwr_null = vbak['NETWR'].isnull().sum()
vbeln_dup = vbak['VBELN'].duplicated().sum()

print("【VBAK 销售订单抬头表】问题检测结果")
print(f"  全行重复: {dup_vbak:,} 条 ({dup_vbak/len(vbak)*100:.3f}%)")
print(f"  净价字段空值（NETWR）: {netwr_null:,} 条")
print(f"  订单号重复: {vbeln_dup:,} 条")
print()

# ── VBAP 问题检测 ──────────────────────────────────────────────
vbap_vbeln_zero = (vbap['VBELN'] == '0000000000').sum()
matnr_null = vbap['MATNR'].isnull().sum()
vbap_dup = vbap.duplicated().sum()

print("【VBAP 销售订单行项目表】问题检测结果")
print(f"  关联失效（订单号=0000000000）: {vbap_vbeln_zero:,} 条 ({vbap_vbeln_zero/len(vbap)*100:.2f}%)")
print(f"  物料编码空值（MATNR）: {matnr_null:,} 条")
print(f"  全行重复: {vbap_dup:,} 条 ({vbap_dup/len(vbap)*100:.3f}%)")

【SAP-ERP】全表质量画像 — KNA1/VBAK/VBAP
KNA1: 15,000 行 | VBAK: 3,015,716 行 | VBAP: 6,060,000 行

【KNA1 客户主数据】问题检测结果
  主键重复（KUNNR）: 0 条 (0.00%)
  营业执照格式错误（STCD1 非18位数字）: 15,000 条 (100.00%)
  名称过短（<4字符）: 0 条
  名称含数字: 0 条
  创建日期越界: 0 条

【VBAK 销售订单抬头表】问题检测结果
  全行重复: 15,028 条 (0.498%)
  净价字段空值（NETWR）: 0 条
  订单号重复: 15,028 条

【VBAP 销售订单行项目表】问题检测结果
  关联失效（订单号=0000000000）: 60,703 条 (1.00%)
  物料编码空值（MATNR）: 0 条
  全行重复: 29,977 条 (0.495%)


#### 3.1.3 SAP-ERP 治理策略

In [4]:
# ── SAP-ERP 治理策略映射 ─────────────────────────────────────────
print("=" * 60)
print("【SAP-ERP】问题识别结果与治理策略")
print("=" * 60)

sap_strategies = [
    {"问题类型": "KNA1: 主键重复（KUNNR）", "实测": f"{dup_kunnr:,} 条", "级别": "🔴 高危",
     "影响": "客户唯一标识冲突 → 销售订单关联错误 → 经营数据失真",
     "治理策略": "去重：按 ERDAT 保留最新记录，其余标记为历史档案",
     "执行层": "ODS → DWD（清洗规则 dedup_kna1）"},
    {"问题类型": "KNA1: 营业执照格式错误", "实测": f"{stcd1_invalid:,} 条", "级别": "🔴 高危",
     "影响": "工商核验接口失效 → 客户准入审核失效 → 合规风险",
     "治理策略": "触发 MDM 主数据修正流程，补录或删除无效记录",
     "执行层": "DQC 规则引擎 → MDM 工单推送"},
    {"问题类型": "KNA1: 客户名称异常", "实测": f"{name_short+name_digit:,} 条", "级别": "🟡 中危",
     "影响": "报表展示混乱 → 下游 BI 分析结果不可信",
     "治理策略": "匹配工商数据库自动修正，人工复核后确认",
     "执行层": "ODS enrichment → DWD 标准命名"},
    {"问题类型": "VBAK: 订单行重复", "实测": f"{dup_vbak:,} 条", "级别": "🟡 中危",
     "影响": "收入统计虚高 → 产销分析数据偏差",
     "治理策略": "按 VBELN+ERDAT 去重，保留最早一条",
     "执行层": "ODS → DWD（清洗规则 dedup_vbak）"},
    {"问题类型": "VBAK: 净价空值", "实测": f"{netwr_null:,} 条", "级别": "🟡 中危",
     "影响": "收入确认不完整 → 财务报表失真",
     "治理策略": "默认值填充（0）+ 推送工单要求业务补录",
     "执行层": "ODS → DWD（默认值 + 问题标记）"},
    {"问题类型": "VBAP: 关联失效（订单号=0）", "实测": f"{vbap_vbeln_zero:,} 条", "级别": "🔴 高危",
     "影响": "行项目孤立 → 无法关联到抬头 → 产销数据缺失",
     "治理策略": "过滤孤立行 + 日志告警 + 数据修正",
     "执行层": "ODS → DWD（过滤规则 filter_invalid_vbap）"},
    {"问题类型": "VBAP: 全行重复", "实测": f"{vbap_dup:,} 条", "级别": "🟡 中危",
     "影响": "发货数量重复统计 → 物流分析偏差",
     "治理策略": "按 VBELN+POSNR 去重",
     "执行层": "ODS → DWD（清洗规则 dedup_vbap）"},
]

df_sap = pd.DataFrame(sap_strategies)
print()
print(df_sap.to_string(index=False))
print()

# 汇总
high = dup_kunnr + stcd1_invalid + vbap_vbeln_zero
med = name_short + name_digit + dup_vbak + netwr_null + vbap_dup
print(f"【治理优先级】🔴 高危: {high:,} 条（阻塞入湖）| 🟡 中危: {med:,} 条（入湖后监控）")
print("→ 结论：VBAP 关联失效（高危）必须先修复，VBAK/VBAP 去重后方可进入 DWD 层。")

【SAP-ERP】问题识别结果与治理策略

             问题类型       实测   级别                           影响                        治理策略                                 执行层
KNA1: 主键重复（KUNNR）      0 条 🔴 高危 客户唯一标识冲突 → 销售订单关联错误 → 经营数据失真 去重：按 ERDAT 保留最新记录，其余标记为历史档案          ODS → DWD（清洗规则 dedup_kna1）
   KNA1: 营业执照格式错误 15,000 条 🔴 高危   工商核验接口失效 → 客户准入审核失效 → 合规风险    触发 MDM 主数据修正流程，补录或删除无效记录                 DQC 规则引擎 → MDM 工单推送
     KNA1: 客户名称异常      0 条 🟡 中危       报表展示混乱 → 下游 BI 分析结果不可信         匹配工商数据库自动修正，人工复核后确认           ODS enrichment → DWD 标准命名
      VBAK: 订单行重复 15,028 条 🟡 中危            收入统计虚高 → 产销分析数据偏差     按 VBELN+ERDAT 去重，保留最早一条          ODS → DWD（清洗规则 dedup_vbak）
       VBAK: 净价空值      0 条 🟡 中危             收入确认不完整 → 财务报表失真        默认值填充（0）+ 推送工单要求业务补录               ODS → DWD（默认值 + 问题标记）
VBAP: 关联失效（订单号=0） 60,703 条 🔴 高危     行项目孤立 → 无法关联到抬头 → 产销数据缺失         过滤孤立行 + 日志告警 + 数据修正 ODS → DWD（过滤规则 filter_invalid_vbap）
       VBAP: 全行重复 29,977 条 🟡 中危            发货数量重复统计 → 物流分析偏差            按 VBELN+POSNR 去重          ODS →

### 3.2 PI-System 表结构说明

**tags — 时序标签数据**

| 字段名 | 含义 | 关键业务含义 |
|--------|------|-------------|
| tag | 传感器标签（全路径） | 命名规则: `{矿井}_{工作面}_{传感器类型}`，如 `M001_FACE_A_WAGAS`，用于唯一标识一个测点 |
| timestamp | 采集时间戳 | 采集时间，1分钟间隔，安全合规要求保存≥2年 |
| value | 测量值 | 传感器测量值，单位随类型变化（WAGAS=%，TEMP=℃，CO=ppm等） |
| status | 状态码 | 采集状态: `0`=正常，`-1`=点位缺失（设备掉线），异常值需结合业务判断 |
| mine | 矿井编码 | 矿井编码（M001-M005），与LIMS/其他系统编码体系不一致，是数据孤岛问题根源之一 |
| face | 工作面编号 | 工作面编号（FACE_A/B/C/D/E），同一矿井有多个工作面 |

**传感器类型速查**

| 类型代码 | 中文名 | 单位 | 安全阈值说明 |
|---------|--------|------|-------------|
| WAGAS | 瓦斯浓度 | % | 重点监控，>0.5%须报警 |
| TEMP | 温度 | ℃ | 一般监控，>40℃设备告警 |
| CO | 一氧化碳 | ppm | 危险气体，>24ppm告警 |
| CO2 | 二氧化碳 | ppm | 窒息气体 |
| PRESS | 压力 | kPa | 设备压力监控 |
| FAN_SPEED | 风机转速 | RPM | 通风系统核心参数 |

#### 3.2.1 PI-System 数据预览

In [5]:
# ── PI-System ──
print("=" * 60)
print("【PI-System】时序数据 (2022年1月样本)")
print("=" * 60)
pi = pd.read_parquet(os.path.join(DATA_ROOT, 'pi_system', 'tags_year=2022_month=01.parquet'))
print(f"记录数: {len(pi):,} | 列: {list(pi.columns)}")
print(f"标签数量: {pi['tag'].nunique()}")
print()
print("标签列表 (前20个):")
print(pi['tag'].unique()[:20])
print()
print("按传感器类型统计:")
pi['sensor_type'] = pi['tag'].str.extract(r'(WAGAS|TEMP|CO|CO2|PRESS|FAN_SPEED)')
print(pi['sensor_type'].value_counts())
print()
print("数据预览:")
print(pi.head(6).to_string(index=False))

【PI-System】时序数据 (2022年1月样本)
记录数: 4,464,000 | 列: ['tag', 'timestamp', 'value', 'status', 'mine', 'face']
标签数量: 100

标签列表 (前20个):
<ArrowStringArray>
['M001_FACE_A_WAGAS',  'M001_FACE_A_TEMP',    'M001_FACE_A_CO',
   'M001_FACE_A_CO2', 'M001_FACE_B_WAGAS',  'M001_FACE_B_TEMP',
    'M001_FACE_B_CO',   'M001_FACE_B_CO2', 'M001_FACE_C_WAGAS',
  'M001_FACE_C_TEMP',    'M001_FACE_C_CO',   'M001_FACE_C_CO2',
 'M001_FACE_D_WAGAS',  'M001_FACE_D_TEMP',    'M001_FACE_D_CO',
   'M001_FACE_D_CO2', 'M001_FACE_E_WAGAS',  'M001_FACE_E_TEMP',
    'M001_FACE_E_CO',   'M001_FACE_E_CO2']
Length: 20, dtype: str

按传感器类型统计:
sensor_type
CO       2232000
WAGAS    1116000
TEMP     1116000
Name: count, dtype: int64

数据预览:
              tag           timestamp   value  status mine        face sensor_type
M001_FACE_A_WAGAS 2022-01-01T00:00:00   0.331       0 M001 M001_FACE_A       WAGAS
 M001_FACE_A_TEMP 2022-01-01T00:01:00  20.391       0 M001 M001_FACE_A        TEMP
   M001_FACE_A_CO 2022-01-01T00:02:00  13.169  

#### 3.2.2 PI-System 质量问题识别

In [8]:
# ── PI-System 质量画像 ──────────────────────────────────────────
print("=" * 60)
print("【PI-System】时序数据质量画像 — 点位缺失 & 异常突升检测")
print("=" * 60)

pi_2023 = pd.read_parquet(os.path.join(DATA_ROOT, 'pi_system', 'tags_year=2023_month=01.parquet'))
print(f"加载 2023-01 样本: {len(pi_2023):,} 条 | 标签数: {pi_2023['tag'].nunique()}")
print()

# 点位缺失检测（status=-1）
missing = (pi_2023['status'] == -1).sum()
missing_pct = missing / len(pi_2023) * 100
print(f"【点位缺失】status=-1 的记录: {missing:,} 条 ({missing_pct:.3f}%)")

# 按矿井统计缺失
pi_2023['sensor_type'] = pi_2023['tag'].str.extract(r'(WAGAS|TEMP|CO|CO2|PRESS|FAN_SPEED)')
pi_2023['mine'] = pi_2023['tag'].str.extract(r'^(M\d+)')
missing_by_mine = pi_2023[pi_2023['status'] == -1].groupby('mine').size()
missing_by_mine.name = '缺失条数'
print("\n按矿井统计缺失:")
print(missing_by_mine.to_string())
print()

# 异常突升检测（WAGAS 超过中位数3倍视为异常）
wagas_data = pi_2023[pi_2023['sensor_type'] == 'WAGAS'].copy()
wagas_median = wagas_data['value'].median()
wagas_p95 = wagas_data['value'].quantile(0.95)
anomaly_mask = (wagas_data['value'] > wagas_median * 3)
anomaly_count = anomaly_mask.sum()
print(f"【WAGAS 异常突升】阈值: 中位数×3 = {wagas_median*3:.3f}%")
print(f"  中位数: {wagas_median:.3f}% | P95: {wagas_p95:.3f}%")
print(f"  异常记录: {anomaly_count:,} 条 ({anomaly_count/len(wagas_data)*100:.3f}%)")

# 按传感器类型统计异常
anomaly_by_type = pi_2023[pi_2023['sensor_type'] == 'WAGAS'].groupby('mine').apply(
    lambda x: (x['value'] > x['value'].median() * 3).sum()
)
print("\n按矿井统计 WAGAS 异常:")
print(anomaly_by_type.to_string())
print()

# 时间连续性检测（是否有时间戳跳跃）
pi_sample = pi_2023[pi_2023['tag'] == pi_2023['tag'].iloc[0]].sort_values('timestamp')
pi_sample['ts'] = pd.to_datetime(pi_sample['timestamp'], errors='coerce')
pi_sample['ts_diff'] = pi_sample['ts'].diff().dt.total_seconds()
gap_count = (pi_sample['ts_diff'] > 120).sum()  # 超过2分钟视为跳跃
print(f"【时间连续性】选取标签 {pi_sample['tag'].iloc[0]}，时间戳跳跃（>2分钟）: {gap_count:,} 次")
print()

# 传感器类型全覆盖检测
print("【传感器类型覆盖】")
print(pi_2023['sensor_type'].value_counts().to_string())
print()

# 问题汇总
print("=" * 60)
print("【PI-System】问题识别与治理策略")
print("=" * 60)
pi_strategies = [
    {"问题类型": "点位缺失（status=-1）", "实测": f"{missing:,} 条 ({missing_pct:.3f}%)",
     "级别": "🟡 中危", "影响": "安全监控存在盲区，瓦斯超标可能被漏采",
     "治理策略": "标记为缺失值，前推/后推填充（线性插值）；发送设备告警工单",
     "执行层": "ODS → DWD（插值填充）+ DQC 告警"},
    {"问题类型": "WAGAS 异常突升", "实测": f"{anomaly_count:,} 条",
     "级别": "🔴 高危", "影响": "可能为传感器故障，数据误采；或真实异常未及时告警",
     "治理策略": "触发安全告警阈值规则，结合前后时间点判断是否为传感器毛刺",
     "执行层": "Kafka 告警规则（Spark Streaming）+ DQC 异常检测"},
    {"问题类型": "时间戳跳跃", "实测": f"{gap_count:,} 次",
     "级别": "🟡 中危", "影响": "合采分析时无法保证时间对齐",
     "治理策略": "标记断点，按传感器类型单独处理时间序列",
     "执行层": "ODS 时间轴切分 + 标签对齐处理"},
]
df_pi = pd.DataFrame(pi_strategies)
print(df_pi.to_string(index=False))
print()
print("→ 结论：WAGAS 异常突升需立即触发安全告警；点位缺失需在入湖时做插值填充，保证时序连续性。")

【PI-System】时序数据质量画像 — 点位缺失 & 异常突升检测
加载 2023-01 样本: 4,464,000 条 | 标签数: 100

【点位缺失】status=-1 的记录: 21,996 条 (0.493%)

按矿井统计缺失:
mine
M001    4402
M002    4383
M003    4434
M004    4445
M005    4332

【WAGAS 异常突升】阈值: 中位数×3 = 1.083%
  中位数: 0.361% | P95: 0.403%
  异常记录: 1,778 条 (0.159%)

按矿井统计 WAGAS 异常:
mine
M001    340
M002    386
M003    367
M004    343
M005    342

【时间连续性】选取标签 M001_FACE_A_WAGAS，时间戳跳跃（>2分钟）: 2,231 次

【传感器类型覆盖】
sensor_type
CO       2232000
WAGAS    1116000
TEMP     1116000

【PI-System】问题识别与治理策略
           问题类型                实测   级别                       影响                          治理策略                                   执行层
点位缺失（status=-1） 21,996 条 (0.493%) 🟡 中危       安全监控存在盲区，瓦斯超标可能被漏采 标记为缺失值，前推/后推填充（线性插值）；发送设备告警工单               ODS → DWD（插值填充）+ DQC 告警
     WAGAS 异常突升           1,778 条 🔴 高危 可能为传感器故障，数据误采；或真实异常未及时告警  触发安全告警阈值规则，结合前后时间点判断是否为传感器毛刺 Kafka 告警规则（Spark Streaming）+ DQC 异常检测
          时间戳跳跃           2,231 次 🟡 中危            合采分析时无法保证时间对齐           标记断点，按传感器类型单独处理时间序列  

#### 3.2.3 PI-System 治理策略

In [9]:
# ── PI-System 问题识别与治理策略 ────────────────────────────────
print("=" * 60)
print("【PI-System】问题识别与治理策略")
print("=" * 60)

pi_strategies = [
    {"问题类型": "点位缺失（status=-1）", "实测": f"{missing:,} 条 ({missing_pct:.3f}%)",
     "级别": "🟡 中危", "影响": "安全监控存在盲区，瓦斯超标可能被漏采",
     "治理策略": "标记为缺失值，前推/后推填充（线性插值）；发送设备告警工单",
     "执行层": "ODS → DWD（插值填充）+ DQC 告警"},
    {"问题类型": "WAGAS 异常突升", "实测": f"{anomaly_count:,} 条",
     "级别": "🔴 高危", "影响": "可能为传感器故障，数据误采；或真实异常未及时告警",
     "治理策略": "触发安全告警阈值规则，结合前后时间点判断是否为传感器毛刺",
     "执行层": "Kafka 告警规则（Spark Streaming）+ DQC 异常检测"},
    {"问题类型": "时间戳跳跃", "实测": f"{gap_count:,} 次",
     "级别": "🟡 中危", "影响": "合采分析时无法保证时间对齐",
     "治理策略": "标记断点，按传感器类型单独处理时间序列",
     "执行层": "ODS 时间轴切分 + 标签对齐处理"},
]
df_pi = pd.DataFrame(pi_strategies)
print(df_pi.to_string(index=False))
print()
print("→ 结论：WAGAS 异常突升需立即触发安全告警；点位缺失需在入湖时做插值填充，保证时序连续性。")


【PI-System】问题识别与治理策略
           问题类型                实测   级别                       影响                          治理策略                                   执行层
点位缺失（status=-1） 21,996 条 (0.493%) 🟡 中危       安全监控存在盲区，瓦斯超标可能被漏采 标记为缺失值，前推/后推填充（线性插值）；发送设备告警工单               ODS → DWD（插值填充）+ DQC 告警
     WAGAS 异常突升           1,778 条 🔴 高危 可能为传感器故障，数据误采；或真实异常未及时告警  触发安全告警阈值规则，结合前后时间点判断是否为传感器毛刺 Kafka 告警规则（Spark Streaming）+ DQC 异常检测
          时间戳跳跃           2,231 次 🟡 中危            合采分析时无法保证时间对齐           标记断点，按传感器类型单独处理时间序列                    ODS 时间轴切分 + 标签对齐处理

→ 结论：WAGAS 异常突升需立即触发安全告警；点位缺失需在入湖时做插值填充，保证时序连续性。


### 3.3 LIMS 表结构说明

**samples — 煤质检测样品表**

| 字段名 | 含义 | 关键业务含义 |
|--------|------|-------------|
| SAMPLE_ID | 样品编号 | 样品唯一编号（主键），用于溯源 |
| MINE_CODE | 矿井编码 | 矿井编码（M001-M005），跨系统关联关键字段，与PI一致但与SAP不同 |
| MINE_NAME | 矿井名称 | 矿井名称，人工录入，存在命名不一致风险 |
| SAMPLE_TYPE | 煤种类型 | 煤种：中煤/原煤/精煤/洗煤/矸石，影响价格定价 |
| SAMPLING_DATE | 采样日期 | 现场采样时间，与化验日期间隔反映样品周转效率 |
| TEST_DATE | 化验日期 | 化验完成时间，用于质量报告时效性分析 |
| TEST_LAB | 化验室 | 化验实验室（一分室/中心化验室），影响结果可比性 |
| REPORTER | 报告人 | 报告人，用于问题追溯 |
| REPORT_STATUS | 报告状态 | 已审核=可对外报送，未审核=待复核 |
| AD | 灰分(%) | 煤中不可燃矿物质含量，越低越好，>35%为劣质煤 |
| VD | 挥发分(%) | 煤在特定条件下析出的气体，与煤化程度相关 |
| FC | 固定碳(%) | 煤中真正可燃烧的碳，与发热量正相关 |
| QGR_AD | 高位发热量(MJ/kg) | 单位质量煤完全燃烧释放的热量，是结算定价核心指标 |
| 全水分Mt | 全水分(%) | 外在水+内在水，越低越好 |
| 全硫St | 全硫(%) | 环保指标，>1%为高硫煤，影响销售价格和环保合规 |
| Mar | 内在水分(%) | 与煤的变质程度相关 |

#### 3.3.1 LIMS 数据预览

In [10]:
# ── LIMS ──
print("=" * 60)
print("【LIMS】煤质检测数据 (2023年样本)")
print("=" * 60)
lims = pd.read_parquet(os.path.join(DATA_ROOT, 'lims', 'samples_year=2023.parquet'))
print(f"记录数: {len(lims):,} | 列: {list(lims.columns)}")
print()
print("煤种分布:")
print(lims['SAMPLE_TYPE'].value_counts())
print()
print("矿井分布:")
print(lims['MINE_CODE'].value_counts())
print()
print("数据预览:")
print(lims.head(3).to_string(index=False))

【LIMS】煤质检测数据 (2023年样本)
记录数: 1,004,535 | 列: ['SAMPLE_ID', 'MINE_CODE', 'MINE_NAME', 'SAMPLE_TYPE', 'SAMPLING_DATE', 'TEST_DATE', 'TEST_LAB', 'REPORTER', 'REPORT_STATUS', 'AD', 'VD', 'FC', 'QGR_AD', '全水分Mt', '全硫St', 'Mar']

煤种分布:
SAMPLE_TYPE
中煤    201442
原煤    201380
精煤    200701
洗煤    200587
矸石    200425
Name: count, dtype: int64

矿井分布:
MINE_CODE
M005    201344
M004    201189
M003    201170
M002    200798
M001    200034
Name: count, dtype: int64

数据预览:
SAMPLE_ID MINE_CODE MINE_NAME SAMPLE_TYPE       SAMPLING_DATE  TEST_DATE TEST_LAB REPORTER REPORT_STATUS    AD    VD    FC  QGR_AD  全水分Mt  全硫St   Mar
 LM861987      M003   朔州安太堡煤矿          中煤 2023-06-01T03:48:30 2023-06-06      一分室       王工           已审核 27.97 30.19 42.14   20.14  11.07  2.25 12.53
 LM542245      M001  鄂尔多斯一号煤矿          原煤 2023-09-07T14:21:35 2023-09-14    中心化验室       孙工           已审核 26.94 25.60 51.19   26.63   8.60  2.03 17.94
 LM683761      M002   榆林李家沟煤矿          原煤 2023-02-08T17:04:01 2023-02-10    中心化验室       刘工    

#### 3.3.2 LIMS 质量问题识别于治理策略

In [11]:
# ── LIMS 质量画像 ───────────────────────────────────────────────
print("=" * 60)
print("【LIMS】煤质检测数据质量画像")
print("=" * 60)

lims_2023 = pd.read_parquet(os.path.join(DATA_ROOT, 'lims', 'samples_year=2023.parquet'))
print(f"加载 2023 样本: {len(lims_2023):,} 条")
print()

# 煤质指标空值检测
quality_cols = ['AD', 'VD', 'FC', 'QGR_AD', '全水分Mt', '全硫St', 'Mar']
null_by_col = lims_2023[quality_cols].isnull().sum()
print("【煤质指标空值检测】")
for col, cnt in null_by_col.items():
    pct = cnt / len(lims_2023) * 100
    print(f"  {col}: {cnt:,} 条 ({pct:.3f}%)")

print()

# 重复行检测（按 SAMPLE_ID）
dup_sample = lims_2023['SAMPLE_ID'].duplicated().sum()
dup_full = lims_2023.duplicated().sum()
print(f"【主键重复检测】SAMPLE_ID 重复: {dup_sample:,} 条")
print(f"【全行重复检测】完全重复: {dup_full:,} 条 ({dup_full/len(lims_2023)*100:.3f}%)")

# 业务逻辑异常检测
lims_2023['SAMPLING_DATE'] = pd.to_datetime(lims_2023['SAMPLING_DATE'], errors='coerce')
lims_2023['TEST_DATE'] = pd.to_datetime(lims_2023['TEST_DATE'], errors='coerce')
lims_2023['TURNAROUND_DAYS'] = (lims_2023['TEST_DATE'] - lims_2023['SAMPLING_DATE']).dt.days

turnaround_slow = (lims_2023['TURNAROUND_DAYS'] > 30).sum()
turnaround_negative = (lims_2023['TURNAROUND_DAYS'] < 0).sum()
print(f"\n【化验时效异常】周转天数>30天: {turnaround_slow:,} 条")
print(f"【逻辑错误】化验日期早于采样日期: {turnaround_negative:,} 条")

# 数值合理性检测（煤质指标范围）
ad_outlier = ((lims_2023['AD'] < 0) | (lims_2023['AD'] > 100)).sum()
qgr_outlier = (lims_2023['QGR_AD'] < 0).sum()
st_outlier = (lims_2023['全硫St'] < 0).sum()
print(f"\n【数值异常】灰分(AD)超出[0,100]%: {ad_outlier:,} 条")
print(f"【数值异常】发热量(QGR_AD)<0: {qgr_outlier:,} 条")
print(f"【数值异常】全硫(St)<0: {st_outlier:,} 条")

# 煤种-指标不一致检测
print("\n【煤种-指标交叉检测】")
for coal_type in lims_2023['SAMPLE_TYPE'].unique():
    subset = lims_2023[lims_2023['SAMPLE_TYPE'] == coal_type]
    ad_mean = subset['AD'].mean()
    print(f"  {coal_type}: 平均灰分={ad_mean:.2f}%")

print()
print("=" * 60)
print("【LIMS】问题识别与治理策略")
print("=" * 60)
lims_strategies = [
    {"问题类型": "样品全行重复", "实测": f"{dup_full:,} 条 ({dup_full/len(lims_2023)*100:.3f}%)",
     "级别": "🟡 中危", "影响": "同一化验结果被统计多次，收入分析偏高",
     "治理策略": "按 SAMPLE_ID 去重，保留最新化验记录",
     "执行层": "ODS → DWD（清洗规则 dedup_lims_samples）"},
    {"问题类型": "化验周转超期（>30天）", "实测": f"{turnaround_slow:,} 条",
     "级别": "🟡 中危", "影响": "样品失效风险，数据过时影响定价准确性",
     "治理策略": "标记为延迟样品，单独统计，不参与实时产销分析",
     "执行层": "ODS → DWD（打标签 turnaround_flag）"},
    {"问题类型": "化验日期早于采样日期", "实测": f"{turnaround_negative:,} 条",
     "级别": "🔴 高危", "影响": "数据录入错误，煤质结果不可信",
     "治理策略": "人工复核，推送 MDM 工单修正",
     "执行层": "DQC 规则告警 → MDM 工单 → 人工处理"},
    {"问题类型": "煤质指标超出合理范围", "实测": f"AD:{ad_outlier:,} St:{st_outlier:,} QGR:{qgr_outlier:,}",
     "级别": "🔴 高危", "影响": "异常数据影响定价，监管报送数据失真",
     "治理策略": "过滤异常值，标记为可疑样品，推送实验室复核",
     "执行层": "ODS → DQC 规则过滤 → DWD 可疑样品表"},
]
df_lims = pd.DataFrame(lims_strategies)
print()
print(df_lims.to_string(index=False))
print()
print("→ 结论：化验日期逻辑错误（高危）必须人工介入；其余问题在 ODS 层做清洗后入 DWD。")

【LIMS】煤质检测数据质量画像
加载 2023 样本: 1,004,535 条

【煤质指标空值检测】
  AD: 0 条 (0.000%)
  VD: 0 条 (0.000%)
  FC: 0 条 (0.000%)
  QGR_AD: 0 条 (0.000%)
  全水分Mt: 0 条 (0.000%)
  全硫St: 0 条 (0.000%)
  Mar: 0 条 (0.000%)

【主键重复检测】SAMPLE_ID 重复: 401,236 条
【全行重复检测】完全重复: 5,081 条 (0.506%)

【化验时效异常】周转天数>30天: 0 条
【逻辑错误】化验日期早于采样日期: 0 条

【数值异常】灰分(AD)超出[0,100]%: 24,965 条
【数值异常】发热量(QGR_AD)<0: 0 条
【数值异常】全硫(St)<0: 0 条

【煤种-指标交叉检测】
  中煤: 平均灰分=49.24%
  原煤: 平均灰分=44.20%
  矸石: 平均灰分=83.26%
  洗煤: 平均灰分=32.33%
  精煤: 平均灰分=28.80%

【LIMS】问题识别与治理策略

        问题类型                   实测   级别                 影响                    治理策略                                执行层
      样品全行重复     5,081 条 (0.506%) 🟡 中危 同一化验结果被统计多次，收入分析偏高 按 SAMPLE_ID 去重，保留最新化验记录 ODS → DWD（清洗规则 dedup_lims_samples）
化验周转超期（>30天）                  0 条 🟡 中危 样品失效风险，数据过时影响定价准确性  标记为延迟样品，单独统计，不参与实时产销分析     ODS → DWD（打标签 turnaround_flag）
  化验日期早于采样日期                  0 条 🔴 高危     数据录入错误，煤质结果不可信        人工复核，推送 MDM 工单修正           DQC 规则告警 → MDM 工单 → 人工处理
  煤质指标超出合理范围 AD:24,965 St:0

### 3.4 OA 表结构说明

**doc_flow — 流程审批记录表**

| 字段名 | 含义 | 关键业务含义 |
|--------|------|-------------|
| FLOW_ID | 流程实例ID | 流程唯一编号（主键） |
| DOC_NO | 文档编号 | 业务文档编号，规则不统一（DOC2022xxx），难以按时间序列分析 |
| FLOW_TYPE | 流程类型 | 请假/报销/付款申请/采购申请/用车申请/出差/公文审批/印章使用，影响流程分析维度 |
| INITIATOR | 申请人 | 申请人姓名，人工录入，存在同音字风险 |
| INITIATOR_DEPT | 申请部门 | 申请部门，是产销分析唯一可用的关联维度（OA无矿井字段） |
| APPLY_DATE | 申请日期 | 申请发起时间，用于流程时效性分析 |
| STATUS | 流程状态 | 已完成/审批中/已驳回/已撤销，影响数据可用性 |
| CURRENT_NODE | 当前审批节点 | 当前审批节点，流程卡在哪一步可追踪 |
| APPROVER | 当前审批人 | 当前审批人，用于流程效率分析 |
| AMOUNT | 申请金额（元） | 付款/采购类金额，金额为空≠0，需区分未填写与金额为0 |

#### 3.4.1 OA 数据预览

In [12]:
# ── OA ──
print("=" * 60)
print("【OA】流程审批数据 (2022年样本)")
print("=" * 60)
oa = pd.read_parquet(os.path.join(DATA_ROOT, 'oa', 'doc_flow_year=2022.parquet'))
print(f"记录数: {len(oa):,} | 列: {list(oa.columns)}")
print()
print("流程类型分布:")
print(oa['FLOW_TYPE'].value_counts())
print()
print("流程状态分布:")
print(oa['STATUS'].value_counts())
print()
print("数据预览:")
print(oa.head(3).to_string(index=False))

【OA】流程审批数据 (2022年样本)
记录数: 2,513,540 | 列: ['FLOW_ID', 'DOC_NO', 'FLOW_TYPE', 'INITIATOR', 'INITIATOR_DEPT', 'APPLY_DATE', 'STATUS', 'CURRENT_NODE', 'APPROVER', 'AMOUNT']

流程类型分布:
FLOW_TYPE
请假      628605
报销      502974
付款申请    377503
采购申请    376196
用车申请    201349
出差      175530
公文审批    125755
印章使用    125628
Name: count, dtype: int64

流程状态分布:
STATUS
已完成    1508226
审批中     628026
已驳回     251684
已撤销     125604
Name: count, dtype: int64

数据预览:
   FLOW_ID       DOC_NO FLOW_TYPE INITIATOR INITIATOR_DEPT          APPLY_DATE STATUS CURRENT_NODE APPROVER   AMOUNT
FL00500001 DOC202200001      付款申请        周涛            采购部 2022-11-30T07:39:40    已完成           归档       张总      NaN
FL00500003 DOC202200003        请假        周涛            技术部 2022-03-12T20:24:14    已驳回         分管领导       王总 10303.27
FL00500004 DOC202200004        请假        吴霞            财务部 2022-02-25T13:56:26    已完成           归档       李总 13921.06


#### 3.4.2 OA 质量问题识别与治理策略

In [13]:
# ── OA 质量画像 ───────────────────────────────────────────────
print("=" * 60)
print("【OA】流程审批数据质量画像")
print("=" * 60)

oa_2022 = pd.read_parquet(os.path.join(DATA_ROOT, 'oa', 'doc_flow_year=2022.parquet'))
print(f"加载 2022 样本: {len(oa_2022):,} 条")
print()

# 重复行检测
dup_flow = oa_2022['FLOW_ID'].duplicated().sum()
dup_full = oa_2022.duplicated().sum()
print(f"【主键重复检测】FLOW_ID 重复: {dup_flow:,} 条")
print(f"【全行重复检测】完全重复: {dup_full:,} 条 ({dup_full/len(oa_2022)*100:.3f}%)")

# 金额字段异常
amount_null = oa_2022['AMOUNT'].isnull().sum()
amount_zero = ((oa_2022['AMOUNT'] == 0) & oa_2022['FLOW_TYPE'].isin(['付款申请', '采购申请'])).sum()
print(f"\n【金额字段】申请金额空值: {amount_null:,} 条")
print(f"【金额异常】付款/采购类申请金额=0: {amount_zero:,} 条")

# 流程状态分布
print(f"\n【流程状态分布】")
print(oa_2022['STATUS'].value_counts().to_string())
in_progress = (oa_2022['STATUS'] == '审批中').sum()
print(f"  → 审批中流程: {in_progress:,} 条（长期搁置风险）")

# 审批时效异常
oa_2022['APPLY_DATE'] = pd.to_datetime(oa_2022['APPLY_DATE'], errors='coerce')
# 简化：用当前年份判断是否超过30天（以时间差衡量）
oa_2022['apply_year'] = oa_2022['APPLY_DATE'].dt.year
# 同一节点停留超过30天的估算（以申请日期到年底的天数）
oa_2022['days_to_year_end'] = (pd.Timestamp(f'{oa_2022["apply_year"].iloc[0]}-12-31') - oa_2022['APPLY_DATE']).dt.days
slow_flow = (oa_2022['days_to_year_end'] > 30).sum()
print(f"【时效异常】申请日期到年底>30天: {slow_flow:,} 条")

# 流程类型-金额交叉检测
print(f"\n【流程类型-金额交叉检测】")
type_amount = oa_2022.groupby('FLOW_TYPE')['AMOUNT'].agg(['count', 'mean', 'max'])
type_amount.columns = ['申请数量', '平均金额', '最大金额']
print(type_amount.to_string())

print()
print("=" * 60)
print("【OA】问题识别与治理策略")
print("=" * 60)
oa_strategies = [
    {"问题类型": "流程全行重复", "实测": f"{dup_full:,} 条 ({dup_full/len(oa_2022)*100:.3f}%)",
     "级别": "🟡 中危", "影响": "同一流程被重复统计，审批效率分析虚高",
     "治理策略": "按 FLOW_ID 去重，保留最新状态",
     "执行层": "ODS → DWD（清洗规则 dedup_oa_flow）"},
    {"问题类型": "付款/采购金额为空", "实测": f"{amount_null:,} 条",
     "级别": "🟡 中危", "影响": "无法统计实际支出，预算执行分析不完整",
     "治理策略": "区分'未填写'与'金额为0'，推送工单要求补录",
     "执行层": "ODS 标签 enrichment → DWD 金额补录标记"},
    {"问题类型": "长期审批中流程（>30天）", "实测": f"{slow_flow:,} 条",
     "级别": "🟡 中危", "影响": "流程卡单未处理，内部控制失效",
     "治理策略": "推送 OA 告警工单至部门负责人，定期清理僵尸流程",
     "执行层": "DQC 流程时效监控 → OA 告警工单"},
    {"问题类型": "审批状态分布不均", "实测": f"审批中{in_progress:,}条",
     "级别": "⚪ 低危", "影响": "需关注异常积压",
     "治理策略": "月度流程健康度报告，持续监控",
     "执行层": "DQC 月度报告 → 管理层 Dashboard"},
]
df_oa = pd.DataFrame(oa_strategies)
print()
print(df_oa.to_string(index=False))
print()
print("→ 结论：OA 数据质量问题相对较轻，主要在 ODS 层做去重和金额补录标记后入 DWD。")

【OA】流程审批数据质量画像
加载 2022 样本: 2,513,540 条

【主键重复检测】FLOW_ID 重复: 12,526 条
【全行重复检测】完全重复: 12,526 条 (0.498%)

【金额字段】申请金额空值: 1,507,345 条
【金额异常】付款/采购类申请金额=0: 0 条

【流程状态分布】
STATUS
已完成    1508226
审批中     628026
已驳回     251684
已撤销     125604
  → 审批中流程: 628,026 条（长期搁置风险）
【时效异常】申请日期到年底>30天: 2,292,810 条

【流程类型-金额交叉检测】
             申请数量          平均金额      最大金额
FLOW_TYPE                                
付款申请       150641  25024.770025  49999.84
公文审批        50155  25037.860920  49999.80
出差          70402  25090.061847  49999.78
印章使用        50325  25046.215889  49999.46
报销         201538  24959.490067  49999.49
用车申请        80672  25063.922217  49999.78
请假         251691  25032.020633  49999.65
采购申请       150771  25065.155273  49999.41

【OA】问题识别与治理策略

         问题类型                实测   级别                 影响                      治理策略                            执行层
       流程全行重复 12,526 条 (0.498%) 🟡 中危 同一流程被重复统计，审批效率分析虚高       按 FLOW_ID 去重，保留最新状态  ODS → DWD（清洗规则 dedup_oa_flow）
    付款/采购金额为空       1,507,345 条 🟡 

### 3.5 SCADA 实时流数据说明与治理策略

> SCADA 数据通过 Kafka 实时流进入数据湖，无历史 Parquet 文件。以下为设计时的**预期问题识别与治理策略**，待实时流启动后可验证。

| 设备类型 | 监控指标 | 数据特征 | 预期质量问题 |
|---------|---------|---------|-------------|
| 皮带机 | 运行状态（0/1） | 高频（秒级） | 状态跳变毛刺 |
| 排水泵 | 流量、压力、温度 | 中频（10秒级） | 传感器漂移 |
| 提升机 | 载重、速度、位置 | 低频（分钟级） | 通信中断丢包 |

In [14]:
# ── SCADA 预期问题识别与治理策略 ─────────────────────────────
print("=" * 60)
print("【SCADA】实时流数据 — 设计时问题识别与治理策略")
print("=" * 60)
print()
print("SCADA 数据特点：")
print("  - 数据来源: Kafka topic (scada_equipment_status)")
print("  - 传输方式: 实时流（秒级），不存储原始历史文件")
print("  - 存储路径: 入湖后以滚动 Parquet 文件写入 data/historical/scada/")
print("  - 设备类型: 皮带机 / 排水泵 / 提升机")
print()

scada_strategies = [
    {"问题类型": "状态跳变毛刺（皮带机）", "预期比例": "~0.5%", "级别": "🟡 中危",
     "影响": "皮带机频繁启停误报，影响生产连续性分析",
     "治理策略": "状态稳定窗口过滤（如：5s内3次跳变视为毛刺）；防抖处理",
     "执行层": "Kafka Streams 实时过滤 → ODS"},
    {"问题类型": "传感器漂移（排水泵/提升机）", "预期比例": "~0.3%", "级别": "🔴 高危",
     "影响": "压力/温度传感器漂移导致设备误报警或漏报警",
     "治理策略": "滑动窗口均值监控，超出±3σ触发告警并冻结数据",
     "执行层": "Flink 实时异常检测 → DQC 告警"},
    {"问题类型": "通信中断丢包", "预期比例": "~0.5%", "级别": "🔴 高危",
     "影响": "设备离线时数据缺失，设备可用率统计不准确",
     "治理策略": "心跳机制检测设备在线状态，离线标记为 NULL，不做插值",
     "执行层": "Kafka 健康检查 → ODS 缺失标记 → DWD"},
    {"问题类型": "时间戳乱序", "预期比例": "~0.1%", "级别": "🟡 中危",
     "影响": "事件顺序错乱，设备故障回放不准确",
     "治理策略": "按设备ID+时间戳排序，写入前做时间轴对齐",
     "执行层": "Flink 事件时间排序（EventTime + Watermark）"},
]
df_scada = pd.DataFrame(scada_strategies)
print("【SCADA】问题识别与治理策略")
print()
print(df_scada.to_string(index=False))
print()
print("→ 结论：SCADA 数据通过 Kafka 入湖，Flink 负责实时清洗和异常检测，DQC 负责设备健康度监控。")
print("  历史数据可通过运行 'uv run python scripts/generate_scada_historical.py' 生成（若需回溯分析）。")

【SCADA】实时流数据 — 设计时问题识别与治理策略

SCADA 数据特点：
  - 数据来源: Kafka topic (scada_equipment_status)
  - 传输方式: 实时流（秒级），不存储原始历史文件
  - 存储路径: 入湖后以滚动 Parquet 文件写入 data/historical/scada/
  - 设备类型: 皮带机 / 排水泵 / 提升机

【SCADA】问题识别与治理策略

          问题类型  预期比例   级别                    影响                         治理策略                                 执行层
   状态跳变毛刺（皮带机） ~0.5% 🟡 中危   皮带机频繁启停误报，影响生产连续性分析 状态稳定窗口过滤（如：5s内3次跳变视为毛刺）；防抖处理            Kafka Streams 实时过滤 → ODS
传感器漂移（排水泵/提升机） ~0.3% 🔴 高危 压力/温度传感器漂移导致设备误报警或漏报警      滑动窗口均值监控，超出±3σ触发告警并冻结数据               Flink 实时异常检测 → DQC 告警
        通信中断丢包 ~0.5% 🔴 高危  设备离线时数据缺失，设备可用率统计不准确 心跳机制检测设备在线状态，离线标记为 NULL，不做插值         Kafka 健康检查 → ODS 缺失标记 → DWD
         时间戳乱序 ~0.1% 🟡 中危      事件顺序错乱，设备故障回放不准确        按设备ID+时间戳排序，写入前做时间轴对齐 Flink 事件时间排序（EventTime + Watermark）

→ 结论：SCADA 数据通过 Kafka 入湖，Flink 负责实时清洗和异常检测，DQC 负责设备健康度监控。
  历史数据可通过运行 'uv run python scripts/generate_scada_historical.py' 生成（若需回溯分析）。


---

## 4. 数据孤岛问题

五个系统使用独立的矿井/客户编码体系，无主外键关联，跨系统分析需要通过主数据映射表实现。同一矿井在不同系统中的名称存在细微差异。

核心数据问题一览

| 问题 | 描述 |
|--------|------|
| 主数据不一致 | SAP 中客户编码6位，PI 中矿井编码无统一格式，LIMS 中矿井编码3位 |
| 时序数据质量问题 | 约0.5%点位缺失（设备掉线），约1%异常突升（传感器故障） |
| 订单-发货不一致 | VBAP 约1%关联到无效订单号 |
| 煤质与订单脱节 | LIMS 批次号与 SAP 物料批次无直接关联 |
| OA 流程数据资产化不足 | 合同编号规则不统一，无法按时间序列分析 |

同一座矿井在不同系统中的编码格式不同，这是数据孤岛的核心表现：

In [15]:
# 矿井编码不一致示例
print("=== 矿井编码不一致问题 ===\n")

# SAP-ERP VBAK 中无 WERKS，改用 VBAP 的 WERKS 字段
vbap_mines = pd.read_parquet(os.path.join(DATA_ROOT, 'sap_erp', 'vbap_year=2023.parquet'))
print("【SAP-ERP VBAP】矿井工厂字段 WERKS 唯一值:")
print(vbap_mines['WERKS'].unique())

print("\n【PI-System】矿井字段 mine 唯一值:")
print(pi['mine'].unique())

print("\n【LIMS】矿井编码字段 MINE_CODE 唯一值:")
print(lims['MINE_CODE'].unique())

print("\n【问题说明】")
print("各系统矿井编码格式不同：")
print("  SAP: WERKS = CN01/CN02/...  →  需映射到 M001/M002/...")
print("  PI:  mine = M001/M002/...     →  标准4位格式")
print("  LIMS: MINE_CODE = M001-M005  →  标准4位格式")
print("  OA:  无矿井字段              →  需通过合同关联")
print("\n这导致跨系统产销分析无法自动关联，需建立主数据映射表。")

=== 矿井编码不一致问题 ===

【SAP-ERP VBAP】矿井工厂字段 WERKS 唯一值:
<ArrowStringArray>
['CN03', 'CN01', 'CN02']
Length: 3, dtype: str

【PI-System】矿井字段 mine 唯一值:
<ArrowStringArray>
['M001', 'M002', 'M003', 'M004', 'M005']
Length: 5, dtype: str

【LIMS】矿井编码字段 MINE_CODE 唯一值:
<ArrowStringArray>
['M003', 'M001', 'M002', 'M005', 'M004']
Length: 5, dtype: str

【问题说明】
各系统矿井编码格式不同：
  SAP: WERKS = CN01/CN02/...  →  需映射到 M001/M002/...
  PI:  mine = M001/M002/...     →  标准4位格式
  LIMS: MINE_CODE = M001-M005  →  标准4位格式
  OA:  无矿井字段              →  需通过合同关联

这导致跨系统产销分析无法自动关联，需建立主数据映射表。


---

## 5. 数据治理四阶段路径

基于以上数据现状，A公司制定了分阶段推进的数据治理路线：

| 阶段 | 目标 | 说明 |
|--------|------|------|
| **Phase 1** | ODS贴源层建设 | 数据采集、CDC入湖、历史数据入湖 |
| **Phase 2** | 主数据管理（MDM） | 矿井、客户、物料编码标准化 |
| **Phase 3** | DWD主题层 + 质量监控 | 清洗规则、质量监控、问题工单 |
| **Phase 4** | 数据资产目录 + 血缘 | 元数据采集、血缘追踪、资产地图 |
| **Phase 5** | DWA应用层 + OLAP | 产销分析、安全预警、即席查询 |

**本 Demo 将依次展示 Phase 1 ~ 5 的核心能力。**

---

## 附1. 数据规模与质量问题注入

本项目生成了约 **1GB** 的模拟数据，覆盖 **2022-01 至 2023-06**（18个月），总记录数约 **1亿行**。所有数据集均按真实场景比例注入了数据质量问题。

In [ ]:
import json
import glob

# 读取元数据
meta = json.load(open(os.path.join(DATA_ROOT, 'metadata.json')))
print("=== 模拟数据生成元数据 ===")
print(f"生成时间: {meta['generated_at']}")
print(f"耗时: {meta['duration_seconds']:.1f} 秒")
print(f"总大小: {meta['total_size_mb']} MB")
print()

# 从实际文件动态读取各系统统计
def scan_parquet_stats(root, system_name, desc):
    files = sorted(glob.glob(os.path.join(root, system_name, '**', '*.parquet'), recursive=True))
    total_rows = 0
    total_size_mb = 0
    for f in files:
        total_size_mb += os.path.getsize(f) / 1024 / 1024
        total_rows += len(pd.read_parquet(f))
    return total_rows, total_size_mb

# SAP-ERP
sap_rows, sap_size = scan_parquet_stats(DATA_ROOT, 'sap_erp', '客户/销售数据')
# PI-System
pi_rows, pi_size = scan_parquet_stats(DATA_ROOT, 'pi_system', '时序传感器数据')
# LIMS
lims_rows, lims_size = scan_parquet_stats(DATA_ROOT, 'lims', '煤质检测数据')
# OA
oa_rows, oa_size = scan_parquet_stats(DATA_ROOT, 'oa', '审批流程记录')

total_rows = sap_rows + pi_rows + lims_rows + oa_rows
total_size = sap_size + pi_size + lims_size + oa_size

# 数据规模表格（动态生成）
scale_info = [
    {"系统": "SAP-ERP", "表/数据集": "KNA1/VBAK/VBAP", "记录数": f"{sap_rows:,}", "存储大小(MB)": f"{sap_size:.1f}", "说明": "客户主数据/销售订单"},
    {"系统": "PI-System", "表/数据集": "tags", "记录数": f"{pi_rows:,}", "存储大小(MB)": f"{pi_size:.1f}", "说明": "100标签时序传感器数据"},
    {"系统": "LIMS", "表/数据集": "samples", "记录数": f"{lims_rows:,}", "存储大小(MB)": f"{lims_size:.1f}", "说明": "煤质检测批次数据"},
    {"系统": "OA", "表/数据集": "doc_flow", "记录数": f"{oa_rows:,}", "存储大小(MB)": f"{oa_size:.1f}", "说明": "审批流程记录"},
    {"系统": "合计", "表/数据集": "", "记录数": f"{total_rows:,}", "存储大小(MB)": f"{total_size:.1f}", "说明": ""},
]

df_scale = pd.DataFrame(scale_info)
print(df_scale.to_string(index=False))

### 1.2 质量问题注入比例

各数据集均按以下比例注入了模拟的数据质量问题，模拟真实生产环境中的数据缺陷：

In [ ]:
# 从实际数据动态检测质量问题注入比例
print("=== 质量问题注入检测（从实际数据反推） ===\n")

quality_info = []

# SAP-ERP: VBAK 空值/重复 + VBAP 关联失效
vbak_v = pd.read_parquet(os.path.join(DATA_ROOT, 'sap_erp', 'vbak_year=2023.parquet'))
vbap_v = pd.read_parquet(os.path.join(DATA_ROOT, 'sap_erp', 'vbap_year=2023.parquet'))
kna1_v = pd.read_parquet(os.path.join(DATA_ROOT, 'sap_erp', 'kna1.parquet'))
null_pct = vbak_v[['NETWR', 'ERZET']].isnull().mean().mean() * 100
dup_pct = vbak_v.duplicated().mean() * 100
invalid_link = (vbap_v['VBELN'] == '0000000000').mean() * 100
quality_info.append({"系统": "SAP-ERP", "质量问题": "空值", "实测比例": f"{null_pct:.3f}%"})
quality_info.append({"系统": "SAP-ERP", "质量问题": "重复行", "实测比例": f"{dup_pct:.3f}%"})
quality_info.append({"系统": "SAP-ERP", "质量问题": "关联失效", "实测比例": f"{invalid_link:.3f}%"})

# PI-System: 点位缺失 + WAGAS 异常突升
pi_v = pd.read_parquet(os.path.join(DATA_ROOT, 'pi_system', 'tags_year=2023_month=01.parquet'))
missing_pct = (pi_v['status'] == -1).mean() * 100
wagas = pi_v[pi_v['tag'].str.contains('WAGAS', na=False)]
wagas_median = wagas['value'].median()
anomaly_pct = (wagas['value'] > wagas_median * 3).mean() * 100
quality_info.append({"系统": "PI-System", "质量问题": "点位缺失", "实测比例": f"{missing_pct:.3f}%"})
quality_info.append({"系统": "PI-System", "质量问题": "WAGAS异常突升", "实测比例": f"{anomaly_pct:.3f}%"})

# LIMS: 空值 + 重复
lims_v = pd.read_parquet(os.path.join(DATA_ROOT, 'lims', 'samples_year=2023.parquet'))
lims_null = lims_v[['AD', 'VD']].isnull().mean().mean() * 100
lims_dup = lims_v.duplicated().mean() * 100
quality_info.append({"系统": "LIMS", "质量问题": "空值", "实测比例": f"{lims_null:.3f}%"})
quality_info.append({"系统": "LIMS", "质量问题": "重复行", "实测比例": f"{lims_dup:.3f}%"})

# OA: 空值 + 重复
oa_v = pd.read_parquet(os.path.join(DATA_ROOT, 'oa', 'doc_flow_year=2023.parquet'))
oa_null = oa_v[['FLOW_TYPE', 'STATUS']].isnull().mean().mean() * 100
oa_dup = oa_v.duplicated().mean() * 100
quality_info.append({"系统": "OA", "质量问题": "空值", "实测比例": f"{oa_null:.3f}%"})
quality_info.append({"系统": "OA", "质量问题": "重复行", "实测比例": f"{oa_dup:.3f}%"})

df_quality = pd.DataFrame(quality_info)
print(df_quality.to_string(index=False))

---

## 附2. 模拟数据生成方法

数据由项目中的 Python 生成器基于向量化方式批量生成：

```bash
# 生成历史数据（~1GB，约3分钟）
uv run python scripts/generate_historical.py

# 生成每日增量
uv run python scripts/generate_incremental.py 2024-01-01 2024-01-31

# 生成 SCADA 实时流（每秒推送Kafka消息）
uv run python -m dg_simulator.scada_simulator
```

生成器采用全向量化实现（NumPy/Pandas），100标签 × 1分钟间隔 × 18个月 ≈ 7862万条时序记录在约2分钟内生成完毕。